## Azure 클라우드 배포하기


### 배포

로컬 환경에서 에이전트를 실행하였으나      
실제 사용자에게 서비스를 제공하려면 **"내 컴퓨터가 꺼져 있어도"**, **"언제 어디서나"**, **"여러 사람이 동시에"** 접속할 수 있어야 함

이를 위해 코드를 클라우드 서버로 옮겨서 24시간 실행되도록 **배포(Deployment)**할 수 있음

### 에이전트 배포의 핵심 구성 요소
1.  **API 서버**: 에이전트와 통신할 수 있는 프로그램 (FastAPI)
2.  **컨테이너 (Container)**: 코드가 어디서든 똑같이 실행되도록 랩핑 (Docker)
3.  **클라우드 플랫폼**: 컨테이너를 실행해주는 컴퓨터 (Azure Container Apps)

### 1. 사전 준비: Azure 로그인

터미널 또는 아래 셀에서 Azure 계정에 로그인합니다.

In [ ]:
# 아래 코드 실행하시면, 인터넷 주소가 코드가 나옵니다.
# 주소를 여시고 코드를 입력하세요. 배정된 MS 계정으로 로그인 하세요!
# 다음 셀부터 실행해 주시면 됩니다. 
!az config set core.login_experience_v2=off
!az login --use-device-code

## 2. 서버 코드 작성 (main.py)

포털에 있는 코드를 차용하여 작성

In [ ]:
import os
import json
import httpx
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Any
from openai import AzureOpenAI

from dotenv import load_dotenv
load_dotenv("../../env")

app = FastAPI(title="Azure AI Agent Service")

# 환경 변수 로드
ENDPOINT = os.getenv("END_POINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")

client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=API_KEY,
    api_version="2024-12-01-preview"
)

def get_weather(city: str) -> str:
    try:
        with httpx.Client() as http_client:
            res = http_client.get("https://geocoding-api.open-meteo.com/v1/search", params={"name": city, "count": 1}).json()
            if not res.get("results"): return f"{city} 검색 실패"
            loc = res["results"][0]
            w_res = http_client.get("https://api.open-meteo.com/v1/forecast", params={
                "latitude": loc["latitude"], "longitude": loc["longitude"], "current_weather": "true"
            }).json()
            return f"{loc['name']} 날씨: {w_res['current_weather']['temperature']}°C"
    except: return "날씨 조회 오류"

class ChatRequest(BaseModel):
    messages: List[Dict[str, Any]]

@app.post("/chat")
async def chat(request: ChatRequest):
    try:
        # 1차 호출 (도구 확인)
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=request.messages,
            tools=[{
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "description": "도시 날씨 확인",
                    "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
                }
            }],
            tool_choice="auto"
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            request.messages.append(msg)
            for tc in msg.tool_calls:
                result = get_weather(json.loads(tc.function.arguments)["city"])
                request.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
            final_res = client.chat.completions.create(model=MODEL_NAME, messages=request.messages)
            return {"response": final_res.choices[0].message.content}
        return {"response": msg.content}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
def root(): return {"status": "ok"}

### 3. 컨테이너 설정 (Dockerfile)

클라우드 환경에서 필요한 패키지가 모두 포함되도록 작성.

```Dockerfile
Dockerfile
FROM python:3.12-slim
WORKDIR /app
RUN pip install --no-cache-dir fastapi uvicorn openai httpx
COPY main.py .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### 4. 배포 및 트러블슈팅

종종 발생하는 **오류**들.

1. **(BadRequest) API version not supported**
   - **원인**: Azure AI Foundry의 게이트웨이가 허용하지 않는 구버전 API를 호출할 때 발생.
   - **해결**: `api_version="2024-12-01-preview"` 처럼 포털에서 권장하는 최신 프리뷰 버전을 명시.

2. **404 Resource Not Found**
   - **원인**: 프로젝트 엔드포인트와 개별 모델 엔드포인트를 혼동하여 주소를 잘못 입력했을 때 발생.
   - **해결**: 포털의 'Azure OpenAI' 섹션에 있는 **Cognitive Services 엔드포인트**를 사용.

3. **500 Internal Server Error**
   - **원인**: 주로 환경 변수(`MODEL_NAME`) 오타나 포트 번호(`8000`) 불일치로 발생.
   - **해결**: `az containerapp up` 시 입력한 변수명이 포털의 **배포 이름**과 완벽히 일치하는지 확인.

### 5. 배포 실행 (Deployment)

아래 정보를 본인의 Azure 리소스 정보로 수정하여 실행합니다.

env 파일에 본인의 id를 넣는걸 잊지 마세요. 다른 사람의 앱을 덮어쓰게 됩니다.

In [ ]:
%%bash
source ../../env
az containerapp up \
  --name foundry-basic-api-${MY_ID} \
  --resource-group rg-KT-new-Foundry \
  --source . \
  --ingress external \
  --target-port 8000 \
  --env-vars \
    END_POINT="$END_POINT" \
    AZURE_OPENAI_API_KEY="$AZURE_OPENAI_API_KEY" \
    MODEL_NAME="gpt-5-mini"


1. name 생성할: Container App 리소스의 고유 식별자
    - 기본적으로 생성되는 FQDN(기본 주소)의 접두어로 사용
1. resource-group: 리소스를 논리적으로 그룹화하는 Azure 리소스 그룹(RG) 이름
    - 권한 관리(RBAC) 및 비용 추적의 기본 단위
1. ingress: 인그레스(수신) 트래픽 설정
    - external은 공용 인터넷 접속을 허용함을 의미. internal로 설정 시 동일한 가상 네트워크 내에서만 접근 가능
1. target-port: 컨테이너 내부에서 애플리케이션이 리스닝(Listening)하는 포트 번호
    - Dockerfile의 EXPOSE 및 Uvicorn/Flask 등의 실행 포트와 반드시 일치해야 함
1. env-vars: 애플리케이션 실행 시 주입되는 런타임 환경 변수,
    - API Endpoint, API Key 등 소스 코드에 하드코딩하면 안 되는 민감 정보를 주입할 때 사용

### 6. 테스트 (Test Remote)

배포된 서비스의 URL을 사용하여 최종 작동 여부를 확인합니다.

In [ ]:
import requests
import subprocess

MY_ID = os.getenv("MY_ID")
APP_NAME = f"foundry-basic-api-{MY_ID}"

fqdn = subprocess.run(
    ["az", "containerapp", "show", "--name", APP_NAME,
    "--resource-group", "rg-KT-new-Foundry",
    "--query", "properties.configuration.ingress.fqdn", "-o", "tsv"],
    capture_output=True, text=True
).stdout.strip()

BASE_URL = f"https://{fqdn}"

def test_agent(query):
    print(f"📤 질문: {query}")
    res = requests.post(f"{BASE_URL}/chat", json={"messages": [{"role": "user", "content": query}]})
    if res.status_code == 200:
        print(f"📥 응답: {res.json()['response']}\n")
    else:
        print(f"❌ 에러: {res.text}")

test_agent("What is the weather in New York?")
test_agent("지금부터 3시간 뒤 서울의 날씨를 알려주세요.")

## 7. 리소스 정리

실습이 종료된 후 비용 과금을 방지하기 위해 앱 삭제

In [ ]:
%%bash
source ../../env
az containerapp delete \
  --name foundry-basic-api-${MY_ID} \
  --resource-group rg-KT-new-Foundry \
  --yes

In [ ]:
%%bash
# 앱의 인스턴스를 0으로 줄여 비용을 최소화 하는 방법 --> 콜드 스타트 발생!
source ../../env
az containerapp update \
  --name foundry-basic-api-${MY_ID} \
  --resource-group rg-KT-new-Foundry \
  --min-replicas 0 \
  --max-replicas 3